# Homework 5 RF Accuracy Improvement

This assignment is inspired by examples of Shan-Hung Wu from National Tsing Hua University.

Requirement: improve the accuracy per feature of the following code from 0.03 up to at least 0.40 and accuracy should be more than 0.92

Here are three hints:

- You can improve the ratio by picking out or "creating" several features.
- Tune hyperparameters
- The ratio can be improved from 0.03 up to 0.47.

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import GridSearchCV

# load the breast_cancer dataset
init_data = load_breast_cancer()
(X, y) = load_breast_cancer(return_X_y=True)
print(X.shape)

(569, 30)


# Determine Feature Importance
In the cell below we find out the importance of each feature.

In [2]:
#split the data between training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.40, random_state = 0)

rfc = RandomForestClassifier(n_estimators = 1000, random_state = 0, n_jobs = -1)
rfc.fit(X_train, y_train)


# Determine importance of each of the 30 features
for feature in zip(init_data.feature_names, rfc.feature_importances_):
    print(feature)


('mean radius', 0.03550745125312651)
('mean texture', 0.010494983839063585)
('mean perimeter', 0.04776880850252026)
('mean area', 0.028827490348813562)
('mean smoothness', 0.005790835323235901)
('mean compactness', 0.01753466609060573)
('mean concavity', 0.07634627800305503)
('mean concave points', 0.13850029138901693)
('mean symmetry', 0.0042308608558732815)
('mean fractal dimension', 0.00338990805041468)
('radius error', 0.01572513079819621)
('texture error', 0.00423302290236439)
('perimeter error', 0.013946544906757447)
('area error', 0.033439564149412765)
('smoothness error', 0.003669991384965381)
('compactness error', 0.005082853553393662)
('concavity error', 0.006534374371210212)
('concave points error', 0.0055915263088285585)
('symmetry error', 0.003321183161356339)
('fractal dimension error', 0.005155422721771359)
('worst radius', 0.08526641825745476)
('worst texture', 0.012370817627080694)
('worst perimeter', 0.1140646944981833)
('worst area', 0.08472316755439184)
('worst smoo

# Determine the most important features
Below we select the features with an importance higher than 0.05

In [3]:
#features that have an importance of more than 0.12
sfm = SelectFromModel(rfc, threshold=0.12)

#Train the selector
sfm.fit(X_train, y_train)

#print the names of the most important features
print('Most Important Features:')
feature_count = 0
for feature_list_index in sfm.get_support(indices=True):
    print(init_data.feature_names[feature_list_index])
    feature_count += 1

Most Important Features:
mean concave points
worst concave points


# Create a new model with only the important features
Train a new model with only the important features as determined by SelectFromModel above.

In [4]:
X_important_train = sfm.transform(X_train)
X_important_test = sfm.transform(X_test)

rfc_important = RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs = -1)
rfc_important.fit(X_important_train, y_train)

RandomForestClassifier(n_estimators=1000, n_jobs=-1, random_state=0)

# Compare accuracy of both models
How accurate are the two models?  Determine the accuracy per feature.

In [5]:
y_pred = rfc.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc}')
print(f'Accuracy per feature: {acc/30}')

Accuracy: 0.9429824561403509
Accuracy per feature: 0.031432748538011694


In [6]:
y_important_pred = rfc_important.predict(X_important_test)
acc = accuracy_score(y_test, y_important_pred)
print(f'Accuracy: {acc}')
print(f'Accuracy per feature: {acc/feature_count}')

Accuracy: 0.9078947368421053
Accuracy per feature: 0.45394736842105265


In [7]:
#Fine tune parameters for RandomForestClassifier

model = RandomForestClassifier()
parameters = {
    "n_estimators": [5, 10, 50, 100, 250, 500, 1000],
    "criterion": ['gini','entropy'],
    "max_depth":[2, 4, 8, 16, 32, None]
}

gscv = GridSearchCV(model, parameters, cv = 5)
gscv.fit(X_important_train, y_train)

print(f'Best parameters are: {gscv.best_params_}')
print("\n")
mean_score = gscv.cv_results_['mean_test_score']
std_score = gscv.cv_results_['std_test_score']
params = gscv.cv_results_['params']
for mean,std, params in zip(mean_score, std_score,params):
    print(f'{round(mean, 3)} + or - {round(std,3)} for the {params}')

Best parameters are: {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 10}


0.921 + or - 0.02 for the {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 5}
0.93 + or - 0.011 for the {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 10}
0.924 + or - 0.017 for the {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 50}
0.921 + or - 0.02 for the {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 100}
0.924 + or - 0.017 for the {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 250}
0.924 + or - 0.017 for the {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 500}
0.921 + or - 0.02 for the {'criterion': 'gini', 'max_depth': 2, 'n_estimators': 1000}
0.909 + or - 0.03 for the {'criterion': 'gini', 'max_depth': 4, 'n_estimators': 5}
0.918 + or - 0.02 for the {'criterion': 'gini', 'max_depth': 4, 'n_estimators': 10}
0.918 + or - 0.02 for the {'criterion': 'gini', 'max_depth': 4, 'n_estimators': 50}
0.918 + or - 0.012 for the {'criterion': 'gini', 'max_depth': 4, 'n_estim

In [22]:
ideal_Model = RandomForestClassifier(n_estimators = gscv.best_params_['n_estimators'], random_state = 1251, max_depth = gscv.best_params_['max_depth'], criterion = gscv.best_params_['criterion'], n_jobs = -1)
ideal_Model.fit(X_important_train, y_train)

y_ideal_pred = ideal_Model.predict(X_important_test)
acc = accuracy_score(y_test, y_ideal_pred)
print(f'Accuracy: {acc}')
print(f'Accuracy per feature: {acc/feature_count}')





Accuracy: 0.9166666666666666
Accuracy per feature: 0.4583333333333333
